# Практика 22 · CLIP і мультимодальність

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ **Зошит навчає пʼятнадцять двобаштових моделей і ще дванадцять
> класифікаторів.** Заміряно двома прогонами на чотирьох ядрах без відеокарти,
> в один потік, на завантаженій машині: **127 і 177 секунд**, тобто дві-три
> хвилини. Найдовша клітинка — замір 4 (девʼять прогонів, 57-73 с).

> 🔌 **Мережа не потрібна.** Ані зображень, ані ваг ми не завантажуємо: фігури
> малюються формулами, описи складаються з їхніх параметрів, обидві вежі
> навчаються тут-таки з випадкових ваг.

Ми зберемо власний крихітний CLIP — дві вежі, спільний простір, симетричну
контрастну втрату — і перевіримо шість тверджень лекції числом.

1. **Замір 1 — спільний простір існує.** Косинус зображення зі своїм описом
   проти косинуса з чужим; матриця схожості на батчі.
2. **Замір 2 — нульовий постріл.** Класифікація без жодного класифікатора,
   проти лінійної проби й проти навчання з нуля.
3. **Замір 3 — композиційність.** Приберемо з навчання комбінацію
   «велике кільце» й подивимось, чи впізнає її модель на тесті.
4. **Замір 4 — температура.** Як вона міняє геометрію спільного простору
   й якість нульового пострілу.
5. **Замір 5 — інженерія підказок.** «Коло» проти «фото кола» проти
   «маленьке коло вгорі».
6. **Замір 6 — де воно ламається.** Заперечення, порядок, лічба.

Плюс **власна реалізація симетричної контрастної втрати** зі звіркою руками
на прикладі 3×3.

In [ ]:
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Один потік. Мережі тут крихітні, і кілька потоків більше домовляються, ніж
# рахують. Друга причина важливіша: під кількома потоками float-суми йдуть в
# іншому порядку, і числа перестають збігатися від прогону до прогону.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Дані: фігури з параметрами

Наскрізний приклад курсу — ті самі шість фігур 28×28. Але тепер у кожної є не
лише клас, а ще два параметри: **розмір** (маленька або велика) і **положення**
в кадрі (у центрі, вгорі, внизу, ліворуч, праворуч).

Шість форм × два розміри × пʼять положень = **60 різних комбінацій**. Саме з
цих параметрів ми потім складемо описи — і саме тому опис нестиме більше
інформації, ніж мітка класу.

Шум σ = 0.45 і дрижання центра ±1 піксель беремо ті самі, що в блоці 3: на
чистіших фігурах міряти нема чого.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]
SHAPE_GEN = ["кола", "квадрата", "ромба", "кільця", "хреста", "трикутника"]
# коло і кільце — середнього роду, решта чоловічого; від цього залежить
# форма прикметника: «велике коло», але «великий квадрат»
NEUTER = [True, False, False, True, False, False]
SIZE_WORDS = [["маленький", "маленьке"], ["великий", "велике"]]
POS_WORDS = ["у центрі", "вгорі", "внизу", "ліворуч", "праворуч"]
POS_OFFSET = [(0, 0), (-5, 0), (5, 0), (0, -5), (0, 5)]
RADIUS = [4, 7]                       # маленька фігура і велика
NOISE = 0.45


def draw_shape(kind, size_index, position_index, rng, noise=NOISE):
    """Малює одну фігуру 28×28 зі значеннями 0..1 за трьома параметрами."""
    image = np.zeros((28, 28), dtype=np.float32)
    radius = RADIUS[size_index]
    shift_y, shift_x = POS_OFFSET[position_index]
    # дрижання ±1 піксель: інакше всі фігури одного класу були б однаковими
    center_y = 14 + shift_y + rng.integers(-1, 2)
    center_x = 14 + shift_x + rng.integers(-1, 2)

    yy, xx = np.mgrid[0:28, 0:28]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                     # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                   # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                   # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                   # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius * 0.5) ** 2)] = 1.0
    elif kind == 4:                                   # хрест
        arm = max(1, radius // 3)
        image[(np.abs(dy) <= arm) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= arm) & (np.abs(dy) <= radius)] = 1.0
    else:                                             # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    if noise > 0:
        image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


# усі 60 комбінацій параметрів — це і є наш «світ»
ALL_COMBOS = [(shape, size, position)
              for shape in range(6) for size in range(2) for position in range(5)]

print("форм     :", len(SHAPE_NAMES))
print("розмірів :", len(RADIUS))
print("положень :", len(POS_WORDS))
print("комбінацій параметрів:", len(ALL_COMBOS))
print("рівень вгадування на шести класах: %.3f" % (1 / 6))

Подивимось на них очима. У верхньому рядку — маленькі фігури, у нижньому —
великі; положення в кадрі різні.

In [ ]:
preview_rng = np.random.default_rng(1)
figure, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for size_index in range(2):
    for shape in range(6):
        position = (shape + size_index * 2) % 5
        picture = draw_shape(shape, size_index, position, preview_rng)
        axes[size_index, shape].imshow(picture, cmap="gray", vmin=0, vmax=1)
        axes[size_index, shape].set_title(
            "%s\n%s" % (SHAPE_NAMES[shape], POS_WORDS[position]), fontsize=8)
        axes[size_index, shape].axis("off")
plt.tight_layout()
plt.show()

print("верхній рядок — маленькі фігури (радіус %d), нижній — великі (радіус %d)"
      % (RADIUS[0], RADIUS[1]))

## 2 · Опис як нагляд

Тепер головний крок теми. Замість мітки «клас 3» ми складаємо з параметрів
**речення природною мовою**: «велике кільце вгорі». Речення несе три факти
замість одного, і жодна людина його не писала — воно вивелось із того самого
генератора, що намалював картинку.

Одна й та сама картинка може мати кілька правильних описів різного ступеня
докладності. Це не проблема, а точна модель реальності: у справжніх даних
підпис під фотографією теж буває і докладним, і однослівним. Заведемо
**пʼять шаблонів**.

In [ ]:
def caption(kind, size_index, position_index, template):
    """Складає опис фігури за її параметрами. Шаблонів пʼять, від голого до повного."""
    adjective = SIZE_WORDS[size_index][1 if NEUTER[kind] else 0]
    if template == 0:
        return SHAPE_NAMES[kind]                                    # «коло»
    if template == 1:
        return "фото " + SHAPE_GEN[kind]                            # «фото кола»
    if template == 2:
        return adjective + " " + SHAPE_NAMES[kind]                  # «велике коло»
    if template == 3:
        return SHAPE_NAMES[kind] + " " + POS_WORDS[position_index]  # «коло вгорі»
    return (adjective + " " + SHAPE_NAMES[kind] + " "
            + POS_WORDS[position_index])                            # «велике коло вгорі»


print("усі пʼять описів однієї й тієї самої картинки (велике кільце вгорі):")
for template in range(5):
    print("  шаблон %d : %s" % (template, caption(3, 1, 1, template)))

print()
print("описи різних комбінацій:")
for combo in [(0, 1, 1), (4, 0, 0), (3, 0, 3), (5, 1, 4)]:
    print("  %-11s %-9s %-10s → «%s»"
          % (SHAPE_NAMES[combo[0]], ["мала", "велика"][combo[1]],
             POS_WORDS[combo[2]], caption(*combo, 4)))

## 3 · Токенізатор: слово → основа

Текстова вежа не вміє читати літери — їй потрібні числа. Найпростіший чесний
спосіб: завести словник і подати опис як **мішок слів** — вектор, у якому
одиниця стоїть проти кожного слова, що трапилось у реченні.

Одна деталь, без якої вся тема розсипалась би. В українській прикметник
змінюється за родом: «велике коло», але «великий квадрат». Якщо вважати ці два
слова різними, то «велике» модель побачить лише з двома формами із шести — і
перевірити композиційність буде неможливо. Тому наш токенізатор ріже слово до
**основи**: «велик-» покриває обидві форми, «кол-» покриває і «коло», і «кола»
з «фото кола». Справжні моделі роблять те саме субслівними токенізаторами.

У словнику є й слова, яких у навчальних описах **не буде ніколи**: «не», «над»,
«під». Вони знадобляться в замірі 6.

In [ ]:
# словник основ. Слово належить основі, якщо починається з неї;
# при кількох збігах виграє найдовша основа
STEMS = ["кол", "квадрат", "ромб", "кільц", "хрест", "трикутник",
         "маленьк", "велик", "центр", "вгор", "вниз", "ліворуч", "праворуч",
         "фото", "не", "над", "під"]
STEM_INDEX = {stem: i for i, stem in enumerate(STEMS)}
VOCAB_SIZE = len(STEMS)


def tokenize(text):
    """Речення → список індексів основ. Невідомі слова просто зникають."""
    indexes = []
    for word in text.lower().replace(",", " ").split():
        best = None
        for stem in STEMS:
            if word.startswith(stem):
                if best is None or len(stem) > len(best):
                    best = stem
        if best is not None:
            indexes.append(STEM_INDEX[best])
    return indexes


def bag_of_words(texts):
    """Список речень → матриця (кількість речень, розмір словника) з нулів і одиниць."""
    matrix = torch.zeros(len(texts), VOCAB_SIZE)
    for row, text in enumerate(texts):
        for index in tokenize(text):
            matrix[row, index] = 1.0
    return matrix


print("розмір словника:", VOCAB_SIZE)
print()
for phrase in ["коло", "фото кола", "велике кільце вгорі",
               "великий квадрат праворуч", "не коло"]:
    print("%-26s → %s" % ("«" + phrase + "»",
                          [STEMS[i] for i in tokenize(phrase)]))

І одразу — перше передбачення, яке ми перевіримо аж у замірі 6. Мішок слів не
знає порядку. Два речення нижче складаються з тих самих трьох основ, тому їхні
мішки збігаються **побітово** — і жодне навчання цього вже не виправить.

In [ ]:
first = bag_of_words(["коло над квадратом"])
second = bag_of_words(["квадрат над колом"])

print("«коло над квадратом» →", [STEMS[i] for i in tokenize("коло над квадратом")])
print("«квадрат над колом»  →", [STEMS[i] for i in tokenize("квадрат над колом")])
print()
print("мішки збігаються побітово:", bool(torch.equal(first, second)))
print("максимальна різниця між векторами: %.1f" % float((first - second).abs().max()))

## 4 · Дві вежі

Тепер сама модель. Вона складається з двох незалежних мереж, які ніде не
перетинаються, крім останнього кроку.

- **Вежа зображень** — знайома згорткова мережа з блоку 2: три блоки
  `Conv → ReLU → Pool` перетворюють 28×28 на 288 чисел, а лінійний шар
  стискає їх до 32.
- **Вежа тексту** — два лінійні шари поверх мішка слів: 17 → 48 → 32.

Головне: **обидві дають вектор однакової довжини — 32 числа**. Тільки тому
їх узагалі можна порівнювати між собою.

In [ ]:
EMBED_DIM = 32


class ImageTower(nn.Module):
    """Зображення 1×28×28 → вектор довжини EMBED_DIM."""

    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten())
        self.projection = nn.Linear(288, dim)

    def forward(self, images):
        return self.projection(self.body(images))


class TextTower(nn.Module):
    """Мішок слів довжини VOCAB_SIZE → вектор довжини EMBED_DIM."""

    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(VOCAB_SIZE, 48), nn.ReLU(), nn.Linear(48, dim))

    def forward(self, bags):
        return self.net(bags)


torch.manual_seed(0)
demo_image_tower, demo_text_tower = ImageTower(), TextTower()
image_params = sum(p.numel() for p in demo_image_tower.parameters())
text_params = sum(p.numel() for p in demo_text_tower.parameters())

# перевіримо, що обидві вежі справді дають вектори однакової довжини
demo_rng = np.random.default_rng(2)
demo_picture = torch.from_numpy(draw_shape(0, 1, 1, demo_rng)[None, None])
with torch.no_grad():
    image_vector = demo_image_tower(demo_picture)
    text_vector = demo_text_tower(bag_of_words(["велике коло вгорі"]))

assert image_vector.shape == text_vector.shape, "вежі дають вектори різної довжини!"

print("ваг у вежі зображень:", image_params)
print("ваг у вежі тексту   :", text_params)
print("вихід вежі зображень:", tuple(image_vector.shape))
print("вихід вежі тексту   :", tuple(text_vector.shape))
print("✅ довжини збігаються — вектори можна порівнювати між собою")

## 5 · Симетрична контрастна втрата — своя реалізація

У [темі 16](../16-self-supervised/lecture.html) ми вже писали контрастну втрату
**NT-Xent**: у чисельнику свій, у знаменнику всі. Тут формула та сама, і це
головна думка теми — **різниця лише в тому, звідки беруться два вектори**. Там
це були два види одного зображення, тут — зображення й текст.

Одна нова деталь. Матриця схожості тепер **не симетрична**: рядки — зображення,
стовпці — описи. Тому втрату рахують двічі: «кожне зображення шукає свій опис»
(по рядках) і «кожен опис шукає своє зображення» (по стовпцях), а потім беруть
середнє. Звідси й назва — симетрична.

Напишемо її самі й звіримо руками на прикладі 3×3.

In [ ]:
def contrastive_loss_from_matrix(similarity, temperature):
    """Симетрична контрастна втрата з готової матриці схожості.

    similarity[i][j] — косинус між i-м зображенням і j-м описом.
    Правильні пари стоять на діагоналі, тому мішень — просто 0, 1, 2, …
    """
    logits = similarity / temperature
    target = torch.arange(len(similarity))
    loss_images_look_for_text = F.cross_entropy(logits, target)      # по рядках
    loss_text_looks_for_images = F.cross_entropy(logits.t(), target)  # по стовпцях
    return 0.5 * (loss_images_look_for_text + loss_text_looks_for_images)


# ── звірка руками, без жодної бібліотечної функції ──
hand_matrix = torch.tensor([[0.90, 0.10, 0.20],
                            [0.00, 0.80, 0.30],
                            [0.15, 0.25, 0.70]])
hand_temperature = 0.5

row_losses = []
for i in range(3):
    own = math.exp(hand_matrix[i][i] / hand_temperature)               # чисельник
    everyone = sum(math.exp(hand_matrix[i][j] / hand_temperature)
                   for j in range(3))                                  # знаменник
    row_losses.append(-math.log(own / everyone))

column_losses = []
for j in range(3):
    own = math.exp(hand_matrix[j][j] / hand_temperature)
    everyone = sum(math.exp(hand_matrix[i][j] / hand_temperature)
                   for i in range(3))
    column_losses.append(-math.log(own / everyone))

hand_loss = 0.5 * (sum(row_losses) / 3 + sum(column_losses) / 3)
library_loss = float(contrastive_loss_from_matrix(hand_matrix, hand_temperature))

print("втрата по рядках  (зображення шукає текст): %.6f" % (sum(row_losses) / 3))
print("втрата по стовпцях (текст шукає зображення): %.6f" % (sum(column_losses) / 3))
print("середнє з двох, порахувати руками : %.6f" % hand_loss)
print("наша функція через cross_entropy  : %.6f" % library_loss)

assert abs(hand_loss - library_loss) < 1e-6, "розрахунок розійшовся!"
print("✅ збігається")
print()
print("а якби модель нічого не знала і всі схожості були однакові,")
print("втрата дорівнювала б log(3) = %.4f" % math.log(3))

Тепер та сама втрата, але схожості рахує сама модель. Вектори обох веж ми
**нормуємо на довжину 1** — після цього скалярний добуток дорівнює косинусу
кута, тобто числу від −1 до 1, яке не залежить від того, наскільки довгі
вектори видала кожна вежа.

In [ ]:
def clip_loss(image_vectors, text_vectors, temperature):
    """Повна втрата CLIP: нормуємо, множимо, рахуємо симетричну контрастну втрату."""
    image_unit = F.normalize(image_vectors, dim=1)
    text_unit = F.normalize(text_vectors, dim=1)
    similarity = image_unit @ text_unit.t()      # рядки — зображення, стовпці — описи
    return contrastive_loss_from_matrix(similarity, temperature)


# на випадкових ненавчених вежах втрата має бути близька до log(розмір батча)
torch.manual_seed(0)
random_image_tower, random_text_tower = ImageTower(), TextTower()
check_rng = np.random.default_rng(3)
check_images = torch.from_numpy(
    np.stack([draw_shape(*ALL_COMBOS[i], check_rng) for i in range(8)])).unsqueeze(1)
check_texts = bag_of_words([caption(*ALL_COMBOS[i], 4) for i in range(8)])

with torch.no_grad():
    start_loss = float(clip_loss(random_image_tower(check_images),
                                 random_text_tower(check_texts), 0.07))

print("втрата на ненавчених вежах: %.4f" % start_loss)
print("стеля повного незнання log(8) = %.4f" % math.log(8))

## 6 · Батч: чому підписи в ньому мусять бути різні

Контрастна втрата вважає **своїм** лише опис на діагоналі, а всі інші описи в
батчі — чужими. Якщо два різні зображення в одному батчі дістануть однаковий
підпис, ми накажемо моделі розсунути те, що насправді однакове. Задача стає
суперечливою.

Тому батч будуємо так: беремо різні комбінації параметрів, для кожної кидаємо
шаблон, а якщо опис уже трапився — падаємо на повний шаблон, який унікальний
для комбінації.

У справжніх даних із інтернету цієї розкоші немає: підписи там повторюються, і
це один із відомих шумів у навчанні CLIP.

In [ ]:
BATCH_SIZE = 32


def make_batch(combos, rng, batch_size=BATCH_SIZE, noise=NOISE):
    """Повертає (зображення, мішки слів) — рівно по одному прикладу на комбінацію."""
    chosen = rng.choice(len(combos), size=min(batch_size, len(combos)), replace=False)
    pictures, texts, already_used = [], [], set()
    for index in chosen:
        shape, size, position = combos[index]
        text = caption(shape, size, position, int(rng.integers(0, 5)))
        if text in already_used:
            text = caption(shape, size, position, 4)   # повний опис унікальний
        if text in already_used:
            continue
        already_used.add(text)
        pictures.append(draw_shape(shape, size, position, rng, noise))
        texts.append(text)
    images = torch.from_numpy(np.stack(pictures)).unsqueeze(1)
    return images, bag_of_words(texts), texts


demo_batch_rng = np.random.default_rng(4)
demo_images, demo_bags, demo_texts = make_batch(ALL_COMBOS, demo_batch_rng)

print("зображень у батчі:", tuple(demo_images.shape))
print("мішків слів      :", tuple(demo_bags.shape))
print("різних описів    :", len(set(demo_texts)), "із", len(demo_texts))
print()
print("перші вісім описів батча:")
for text in demo_texts[:8]:
    print("  «%s»" % text)

## 7 · Навчання

Півтори сотні рядків позаду — лишилось крутити цикл. Оптимізатор **AdamW**,
швидкість навчання 3·10⁻³, 500 кроків по 32 пари, температура 0.07 (саме таке
значення бере справжній CLIP на старті).

Навчаємо **три моделі з різними зернами**: розкид від зерна в цьому курсі
сягав 0.33, і одне число нічого не доводить.

In [ ]:
TEMPERATURE = 0.07
TRAIN_STEPS = 500


def train_clip(combos, seed, steps=TRAIN_STEPS, temperature=TEMPERATURE,
               learning_rate=3e-3, noise=NOISE):
    """Навчає пару веж на парах «зображення — опис». Повертає обидві вежі."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(1000 + seed)
    image_tower, text_tower = ImageTower(), TextTower()
    optimizer = torch.optim.AdamW(
        list(image_tower.parameters()) + list(text_tower.parameters()),
        lr=learning_rate)

    history = []
    for step in range(steps):
        images, bags, _ = make_batch(combos, rng, noise=noise)
        loss = clip_loss(image_tower(images), text_tower(bags), temperature)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach()))
    return image_tower, text_tower, history


training_started = time.perf_counter()
models = []
for seed in (0, 1, 2):
    image_tower, text_tower, history = train_clip(ALL_COMBOS, seed)
    models.append((image_tower, text_tower))
    print("зерно %d: втрата з %.3f впала до %.3f" % (seed, history[0], history[-1]))

print()
print("три моделі навчено за %.0f секунд" % (time.perf_counter() - training_started))
print("стеля повного незнання для батча 32: log(32) = %.3f" % math.log(32))

## 8 · Замір 1: спільний простір існує

Перше й найголовніше твердження: після навчання зображення лежить **поруч зі
своїм описом і далеко від чужих**. Перевіримо це прямо — візьмемо дванадцять
тестових зображень, складемо для кожного правильний повний опис і порахуємо
матрицю косинусів.

Цікавлять три числа: середній косинес на діагоналі, середній косинус поза нею
і частка рядків, у яких найбільше число стоїть саме на діагоналі (це і є
**пошук зображення за текстом** — окрема корисна задача сама по собі).

In [ ]:
def make_test_set(combos, per_combo, rng, noise=NOISE):
    """Тестовий набір: однаково по per_combo прикладів на кожну комбінацію."""
    pictures, meta = [], []
    for combo in combos:
        for _ in range(per_combo):
            pictures.append(draw_shape(combo[0], combo[1], combo[2], rng, noise))
            meta.append(combo)
    return torch.from_numpy(np.stack(pictures)).unsqueeze(1), meta


test_rng = np.random.default_rng(777)
x_test, meta_test = make_test_set(ALL_COMBOS, 10, test_rng)
y_shape = torch.tensor([m[0] for m in meta_test])

print("тестових зображень:", tuple(x_test.shape))
print("по %d на кожну з %d комбінацій" % (10, len(ALL_COMBOS)))


@torch.no_grad()
def embed_images(image_tower, images):
    return F.normalize(image_tower(images), dim=1)


@torch.no_grad()
def embed_texts(text_tower, texts):
    return F.normalize(text_tower(bag_of_words(texts)), dim=1)

In [ ]:
def similarity_report(image_tower, text_tower, how_many=12):
    """Матриця «зображення × опис» на випадковій вибірці тестових прикладів."""
    chosen = np.random.default_rng(5).choice(len(meta_test), how_many, replace=False)
    picture_vectors = embed_images(image_tower, x_test[chosen])
    text_vectors = embed_texts(text_tower, [caption(*meta_test[i], 4) for i in chosen])
    matrix = (picture_vectors @ text_vectors.t()).numpy()
    on_diagonal = float(np.diag(matrix).mean())
    off_diagonal = float((matrix.sum() - np.trace(matrix)) / (how_many * how_many - how_many))
    retrieval = float((matrix.argmax(1) == np.arange(how_many)).mean())
    return on_diagonal, off_diagonal, retrieval, matrix


diagonals, off_diagonals, retrievals = [], [], []
for image_tower, text_tower in models:
    on, off, retrieval, _ = similarity_report(image_tower, text_tower)
    diagonals.append(on)
    off_diagonals.append(off)
    retrievals.append(retrieval)


def show(name, values):
    array = np.array(values)
    print("%-42s %.3f ±%.3f   %s" % (name, array.mean(), array.std(), np.round(array, 3)))
    return array.mean(), array.std()


show("косинус зі СВОЇМ описом", diagonals)
show("косинус із ЧУЖИМ описом", off_diagonals)
show("пошук опису за зображенням (з 12)", retrievals)
print()
print("рівень вгадування при пошуку з 12: %.3f" % (1 / 12))

Намалюємо цю матрицю. Діагональ має світитися — і саме так виглядає картинка,
яку в статті CLIP показують першою.

In [ ]:
_, _, _, matrix_to_draw = similarity_report(models[0][0], models[0][1], how_many=8)

plt.figure(figsize=(5.2, 4.4))
plt.imshow(matrix_to_draw, cmap="magma", vmin=-0.3, vmax=1.0)
plt.colorbar(label="косинус")
plt.xlabel("описи")
plt.ylabel("зображення")
plt.title("матриця схожості 8×8")
plt.tight_layout()
plt.show()

print("діагональ         :", np.round(np.diag(matrix_to_draw), 3))
print("максимум поза нею : %.3f" % float(
    (matrix_to_draw - np.eye(8) * 10).max()))
print("середнє поза нею  : %.3f" % float(
    (matrix_to_draw.sum() - np.trace(matrix_to_draw)) / 56))

## 9 · Замір 2: нульовий постріл

Ось нова здатність, якої в курсі ще не було. Щоб класифікувати зображення, нам
**не потрібен класифікатор**. Досить написати шість описів — по одному на клас —
прогнати їх через текстову вежу й спитати, до якого з шести векторів зображення
ближче.

Жодної мітки при цьому не використано: ані на навчанні (там були описи, а не
класи), ані на класифікації.

In [ ]:
PROMPT_SETS = {
    "гола назва": SHAPE_NAMES,
    "фото X": ["фото " + word for word in SHAPE_GEN],
    "надто конкретний": [SIZE_WORDS[0][1 if NEUTER[k] else 0] + " "
                         + SHAPE_NAMES[k] + " вгорі" for k in range(6)],
}


def zero_shot_accuracy(image_tower, text_tower, prompts):
    """Класифікація без класифікатора: до якого з описів зображення ближче."""
    prompt_vectors = embed_texts(text_tower, prompts)
    picture_vectors = embed_images(image_tower, x_test)
    predicted = (picture_vectors @ prompt_vectors.t()).argmax(1)
    return float((predicted == y_shape).float().mean())


zero_shot_results = {}
for name, prompts in PROMPT_SETS.items():
    zero_shot_results[name] = [zero_shot_accuracy(i, t, prompts) for i, t in models]
    show("нульовий постріл · " + name, zero_shot_results[name])
print()
print("рівень вгадування: %.3f" % (1 / 6))

Опис знає більше, ніж клас, — і це видно. Спитаємо модель не «яка це фігура», а
«вона велика чи маленька» і «де вона в кадрі». Класифікатора для цих питань у
нас ніколи не було й не буде: класи задаються текстом на льоту.

In [ ]:
y_size = torch.tensor([m[1] for m in meta_test])
y_position = torch.tensor([m[2] for m in meta_test])


def zero_shot_attribute(image_tower, text_tower, which):
    """Нульовий постріл про розмір або положення: форму в підказці називаємо правильно."""
    picture_vectors = embed_images(image_tower, x_test)
    correct = 0
    for i, (shape, size, position) in enumerate(meta_test):
        if which == "size":
            prompts = [SIZE_WORDS[s][1 if NEUTER[shape] else 0] + " " + SHAPE_NAMES[shape]
                       for s in range(2)]
            answer = size
        else:
            prompts = [SHAPE_NAMES[shape] + " " + POS_WORDS[p] for p in range(5)]
            answer = position
        prompt_vectors = embed_texts(text_tower, prompts)
        correct += int((picture_vectors[i] @ prompt_vectors.t()).argmax().item() == answer)
    return correct / len(meta_test)


show("нульовий постріл про розмір (2 варіанти)",
     [zero_shot_attribute(i, t, "size") for i, t in models])
show("нульовий постріл про положення (5 варіантів)",
     [zero_shot_attribute(i, t, "position") for i, t in models])
print()
print("рівень вгадування: розмір %.3f, положення %.3f" % (0.5, 0.2))

Тепер порівняння, заради якого все й затівалось. Поставимо поруч три способи
дістати відповідь «яка це фігура»:

- **нульовий постріл** — нуль міток;
- **лінійна проба** (та сама, що в [темі 16](../16-self-supervised/lecture.html)):
  вежу зображень заморожуємо, згори ставимо один лінійний шар і вчимо його на
  60 або 300 мітках;
- **навчання з нуля** — та сама згорткова мережа з випадкових ваг на тих самих мітках.

In [ ]:
def make_labeled_set(count, seed):
    """Позначений набір: рівномірно по всіх комбінаціях, мітка — форма."""
    rng = np.random.default_rng(seed)
    pictures, labels = [], []
    for i in range(count):
        combo = ALL_COMBOS[i % len(ALL_COMBOS)]
        pictures.append(draw_shape(combo[0], combo[1], combo[2], rng))
        labels.append(combo[0])
    return torch.from_numpy(np.stack(pictures)).unsqueeze(1), torch.tensor(labels)


def linear_probe(image_tower, count, seed):
    """Заморожена вежа плюс один лінійний шар, навчений на count мітках."""
    torch.manual_seed(seed)
    x_train, y_train = make_labeled_set(count, 3000 + seed)
    with torch.no_grad():
        train_features = embed_images(image_tower, x_train)
        test_features = embed_images(image_tower, x_test)
    head = nn.Linear(EMBED_DIM, 6)
    optimizer = torch.optim.AdamW(head.parameters(), lr=3e-2)
    for _ in range(200):
        loss = F.cross_entropy(head(train_features), y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        return float((head(test_features).argmax(1) == y_shape).float().mean())


def train_from_scratch(count, seed):
    """Звичайний класифікатор із випадкових ваг: те саме тіло, той самий бюджет кроків."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(4000 + seed)
    x_train, y_train = make_labeled_set(count, 4000 + seed)
    net = ImageTower(6)                       # та сама мережа, але вихід — шість класів
    optimizer = torch.optim.AdamW(net.parameters(), lr=3e-3)
    for _ in range(TRAIN_STEPS):
        batch = torch.from_numpy(rng.choice(count, size=min(32, count), replace=False))
        loss = F.cross_entropy(net(x_train[batch]), y_train[batch])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        return float((net(x_test).argmax(1) == y_shape).float().mean())


baseline_started = time.perf_counter()
print("%-24s %-18s %s" % ("міток", "лінійна проба", "з нуля"))
print("-" * 56)
for count in (60, 300):
    probe = np.array([linear_probe(models[i][0], count, seed)
                      for i, seed in enumerate((0, 1, 2))])
    scratch = np.array([train_from_scratch(count, seed) for seed in (0, 1, 2)])
    print("%-24d %.3f ±%.3f       %.3f ±%.3f"
          % (count, probe.mean(), probe.std(), scratch.mean(), scratch.std()))

best_zero_shot = np.array(zero_shot_results["фото X"])
print("-" * 56)
print("%-24s %.3f ±%.3f" % ("нульовий постріл, 0 міток",
                            best_zero_shot.mean(), best_zero_shot.std()))
print()
print("порівняння зайняло %.0f с" % (time.perf_counter() - baseline_started))

## 10 · Замір 3: композиційність

Найцікавіше питання теми. Модель бачила «велике коло» і бачила «маленьке
кільце». Чи впізнає вона **велике кільце**, якщо цієї пари в навчанні не було
жодного разу?

Якщо так — вона склала два слова, яких разом не бачила, і це справжня
композиційність. Якщо ні — теж результат, і його треба назвати прямо.

Заберемо з навчання всі пʼять комбінацій «кільце × великий розмір» і навчимо
три нові моделі. Перевірятимемо на 12 підказках виду «розмір + форма»: модель
має вибрати «велике кільце».

In [ ]:
HELD_OUT_SHAPE, HELD_OUT_SIZE = 3, 1          # кільце, великий розмір
train_combos_without = [c for c in ALL_COMBOS
                        if not (c[0] == HELD_OUT_SHAPE and c[1] == HELD_OUT_SIZE)]

PAIR_PROMPTS = [SIZE_WORDS[s][1 if NEUTER[k] else 0] + " " + SHAPE_NAMES[k]
                for k in range(6) for s in range(2)]
PAIR_KEY = [(k, s) for k in range(6) for s in range(2)]

held_out_indexes = [i for i, m in enumerate(meta_test)
                    if m[0] == HELD_OUT_SHAPE and m[1] == HELD_OUT_SIZE]
seen_indexes = [i for i, m in enumerate(meta_test)
                if not (m[0] == HELD_OUT_SHAPE and m[1] == HELD_OUT_SIZE)]

print("комбінацій у навчанні було :", len(ALL_COMBOS))
print("комбінацій лишилось        :", len(train_combos_without))
print("прибрано                   :", len(ALL_COMBOS) - len(train_combos_without),
      "(«велике кільце» в усіх пʼятьох положеннях)")
print("підказок для перевірки     :", len(PAIR_PROMPTS),
      "· рівень вгадування %.3f" % (1 / len(PAIR_PROMPTS)))
print("тестових великих кілець    :", len(held_out_indexes))

In [ ]:
holdout_started = time.perf_counter()
models_without = []
for seed in (0, 1, 2):
    image_tower, text_tower, history = train_clip(train_combos_without, 100 + seed)
    models_without.append((image_tower, text_tower))
    print("зерно %d: втрата з %.3f впала до %.3f" % (seed, history[0], history[-1]))

print()
print("три моделі без «велике кільце» навчено за %.0f с"
      % (time.perf_counter() - holdout_started))

In [ ]:
def pair_accuracy(image_tower, text_tower, indexes):
    """Три числа: скільки разів угадано пару цілком, скільки — форму, скільки — розмір."""
    prompt_vectors = embed_texts(text_tower, PAIR_PROMPTS)
    picture_vectors = embed_images(image_tower, x_test[indexes])
    predicted = (picture_vectors @ prompt_vectors.t()).argmax(1)
    both = shape_ok = size_ok = 0
    for row, i in enumerate(indexes):
        shape, size = PAIR_KEY[predicted[row].item()]
        shape_ok += int(shape == meta_test[i][0])
        size_ok += int(size == meta_test[i][1])
        both += int(shape == meta_test[i][0] and size == meta_test[i][1])
    return both / len(indexes), shape_ok / len(indexes), size_ok / len(indexes)


print("%-34s %-16s %-14s %s" % ("модель і набір", "пара цілком", "форма", "розмір"))
print("-" * 80)
composition_numbers = {}
for tag, group in (("БЕЗ «велике кільце»", models_without), ("З «велике кільце»", models)):
    for where, indexes in (("великі кільця", held_out_indexes), ("бачені пари", seen_indexes)):
        rows = np.array([pair_accuracy(i, t, indexes) for i, t in group])
        composition_numbers[(tag, where)] = rows
        print("%-34s %.3f ±%.3f    %.3f ±%.3f   %.3f ±%.3f"
              % (tag + " → " + where,
                 rows[:, 0].mean(), rows[:, 0].std(),
                 rows[:, 1].mean(), rows[:, 1].std(),
                 rows[:, 2].mean(), rows[:, 2].std()))
print("-" * 80)
print("рівень вгадування: пара %.3f, форма %.3f, розмір %.3f"
      % (1 / 12, 1 / 6, 1 / 2))

А тепер найважливіше — подивитись, **куди саме** показує модель, яка не бачила
великих кілець. Порахуємо, які підказки вона обирає для великого кільця.

In [ ]:
image_tower, text_tower = models_without[0]
prompt_vectors = embed_texts(text_tower, PAIR_PROMPTS)
picture_vectors = embed_images(image_tower, x_test[held_out_indexes])
predicted = (picture_vectors @ prompt_vectors.t()).argmax(1)

counts = {}
for index in predicted.tolist():
    counts[PAIR_PROMPTS[index]] = counts.get(PAIR_PROMPTS[index], 0) + 1

print("що модель без «велике кільце» каже про велике кільце (%d прикладів):"
      % len(held_out_indexes))
for text, number in sorted(counts.items(), key=lambda pair: -pair[1]):
    print("  %-22s %d" % ("«" + text + "»", number))

big_words = sum(number for text, number in counts.items() if text.startswith("велик"))
print()
print("з них зі словом «велик-»: %d із %d (%.3f)"
      % (big_words, len(held_out_indexes), big_words / len(held_out_indexes)))

Один приклад крупним планом: усі дванадцять косинусів для одного великого
кільця, у двох моделей — тієї, що бачила цю пару, і тієї, що не бачила.

In [ ]:
example_rng = np.random.default_rng(31)
ring_picture = draw_shape(HELD_OUT_SHAPE, HELD_OUT_SIZE, 0, example_rng)
ring_tensor = torch.from_numpy(ring_picture[None, None])

cosines_without = (embed_images(models_without[0][0], ring_tensor)
                   @ embed_texts(models_without[0][1], PAIR_PROMPTS).t()).numpy()[0]
cosines_with = (embed_images(models[0][0], ring_tensor)
                @ embed_texts(models[0][1], PAIR_PROMPTS).t()).numpy()[0]

print("%-22s %-14s %s" % ("підказка", "БЕЗ пари", "З парою"))
print("-" * 52)
for text, without, with_pair in zip(PAIR_PROMPTS, cosines_without, cosines_with):
    mark = " ←  правильна" if text == "велике кільце" else ""
    print("%-22s %+.3f         %+.3f%s" % ("«" + text + "»", without, with_pair, mark))

## 11 · Замір 4: температура

Температура τ (грецька «тау» — просто число) ділить усі схожості перед
експонентою. У [темі 16](../16-self-supervised/lecture.html) ми бачили, що вона
вирішує, кого модель слухає: мала температура робить втрату майже повністю
залежною від найсхожішого чужого прикладу.

Тут подивимось на неї з іншого боку — **як вона формує сам спільний простір**.
Навчимо моделі з чотирма різними температурами й порівняємо три речі: косинус
на діагоналі, косинус поза діагоналлю і точність нульового пострілу.

In [ ]:
temperature_started = time.perf_counter()
print("%-8s %-16s %-16s %s" % ("τ", "свій опис", "чужий опис", "нульовий постріл"))
print("-" * 62)

temperature_table = {}
for temperature in (0.02, 0.07, 0.20, 0.50):
    accuracies, on_diagonal, off_diagonal = [], [], []
    for seed in (0, 1, 2):
        if abs(temperature - TEMPERATURE) < 1e-9:
            image_tower, text_tower = models[seed]      # ці вже навчені вище
        else:
            image_tower, text_tower, _ = train_clip(ALL_COMBOS, seed,
                                                    temperature=temperature)
        on, off, _, _ = similarity_report(image_tower, text_tower)
        on_diagonal.append(on)
        off_diagonal.append(off)
        accuracies.append(zero_shot_accuracy(image_tower, text_tower,
                                             PROMPT_SETS["фото X"]))
    temperature_table[temperature] = (np.mean(on_diagonal), np.mean(off_diagonal),
                                      np.mean(accuracies), np.std(accuracies))
    print("%-8.2f %+.3f           %+.3f           %.3f ±%.3f"
          % (temperature, np.mean(on_diagonal), np.mean(off_diagonal),
             np.mean(accuracies), np.std(accuracies)))

print("-" * 62)
print("замір зайняв %.0f с" % (time.perf_counter() - temperature_started))

## 12 · Замір 5: інженерія підказок

Класи в нульовому пострілі задаються **текстом**, а текст можна написати
по-різному. Порівняємо чотири способи назвати ті самі шість класів. Дивитись
треба не на середні, а на **різницю в межах одного зерна**: розкид від зерна
більший за різницю між підказками, і середні самі по собі нічого не доведуть.

In [ ]:
def prompt_ensemble(image_tower, text_tower, prompts_by_class):
    """Ансамбль підказок: усереднюємо вектори кількох формулювань одного класу."""
    picture_vectors = embed_images(image_tower, x_test)
    columns = []
    for shape in range(6):
        vectors = embed_texts(text_tower, prompts_by_class[shape])
        columns.append(F.normalize(vectors.mean(0), dim=0))
    predicted = (picture_vectors @ torch.stack(columns).t()).argmax(1)
    return float((predicted == y_shape).float().mean())


# два формулювання на клас: гола назва й «фото …»
two_variants = [[SHAPE_NAMES[k], "фото " + SHAPE_GEN[k]] for k in range(6)]

variants = {
    "«коло»": zero_shot_results["гола назва"],
    "«фото кола»": zero_shot_results["фото X"],
    "«маленьке коло вгорі»": zero_shot_results["надто конкретний"],
    "ансамбль двох формулювань": [prompt_ensemble(i, t, two_variants) for i, t in models],
}

print("%-30s %-18s %s" % ("формулювання", "точність", "по зернах"))
print("-" * 68)
for name, values in variants.items():
    array = np.array(values)
    print("%-30s %.3f ±%.3f      %s" % (name, array.mean(), array.std(), np.round(array, 3)))

print("-" * 68)
base = np.array(variants["«коло»"])
print("різниця з «голою назвою» в межах кожного зерна:")
for name, values in variants.items():
    if name == "«коло»":
        continue
    difference = np.array(values) - base
    sign = "завжди краще" if (difference > 0).all() else (
        "завжди гірше" if (difference < 0).all() else "по-різному")
    print("  %-30s %s   → %s" % (name, np.round(difference, 3), sign))

## 13 · Замір 6: де воно ламається

Три речі, яких контрастна модель не вміє в принципі. Не «вміє погано» — саме
не вміє, і ми зараз побачимо чому.

**Заперечення.** Дамо моделі шість підказок «не коло», «не квадрат» і так далі.
Правильною для зображення кола є будь-яка, крім «не коло». Порахуємо, як часто
модель обирає рівно ту єдину, що неправильна.

In [ ]:
negation_prompts = ["не " + name for name in SHAPE_NAMES]

negation_results = [zero_shot_accuracy(i, t, negation_prompts) for i, t in models]
show("«не X» вказує на власний клас зображення", negation_results)
print()
print("тобто модель обирає ЄДИНУ підказку, яка точно хибна.")
print("для порівняння, та сама модель на звичайних підказках: %.3f"
      % np.mean(zero_shot_results["гола назва"]))
print()

circle = embed_texts(models[0][1], ["коло"])[0]
not_circle = embed_texts(models[0][1], ["не коло"])[0]
print("косинус між «коло» і «не коло»: %.4f" % float(circle @ not_circle))
print("слово «не» додає до мішка одну одиницю — і більше нічого;")
print("механізму, який перевертав би зміст, у моделі немає.")

**Порядок.** Ми вже бачили, що мішки слів двох речень про порядок збігаються
побітово. Подивимось, що з цього виходить після навчання: різниця косинусів
для двофігурного зображення.

In [ ]:
def draw_two_shapes(top_kind, bottom_kind, rng):
    """Дві маленькі фігури в одному кадрі: одна вгорі, друга внизу."""
    top = draw_shape(top_kind, 0, 1, rng, noise=0.0)
    bottom = draw_shape(bottom_kind, 0, 2, rng, noise=0.0)
    together = np.clip(top + bottom, 0, 1)
    together = together + rng.normal(0, NOISE, together.shape).astype(np.float32)
    return np.clip(together, 0, 1)


two_rng = np.random.default_rng(41)
two_picture = draw_two_shapes(0, 1, two_rng)        # коло вгорі, квадрат унизу

plt.figure(figsize=(2.4, 2.4))
plt.imshow(two_picture, cmap="gray", vmin=0, vmax=1)
plt.title("коло над квадратом", fontsize=10)
plt.axis("off")
plt.show()

order_prompts = ["коло над квадратом", "квадрат над колом"]
order_cosines = (embed_images(models[0][0], torch.from_numpy(two_picture[None, None]))
                 @ embed_texts(models[0][1], order_prompts).t()).numpy()[0]

print("косинус з «коло над квадратом»: %.6f" % order_cosines[0])
print("косинус з «квадрат над колом» : %.6f" % order_cosines[1])
print("різниця                        : %.6f" % abs(order_cosines[0] - order_cosines[1]))
print()
first_text = embed_texts(models[0][1], ["коло над квадратом"])[0]
second_text = embed_texts(models[0][1], ["квадрат над колом"])[0]
print("косинус між самими описами: %.6f" % float(first_text @ second_text))
print("це один і той самий вектор — модель фізично не може їх розрізнити.")

**Лічба.** У нашому словнику немає жодного числівника. «Два кола» перетворюється
на той самий мішок, що й «коло», — модель не бачить різниці між одним предметом
і двома.

In [ ]:
for phrase in ["коло", "два кола", "три кола"]:
    print("%-14s → %s" % ("«" + phrase + "»", [STEMS[i] for i in tokenize(phrase)]))

one = embed_texts(models[0][1], ["коло"])[0]
two = embed_texts(models[0][1], ["два кола"])[0]
print()
print("косинус між «коло» і «два кола»: %.6f" % float(one @ two))
print("кількість не виражена нічим, тому й розрізнити її нічим.")

## 14 · Що вийшло

Шість замірів, усі числа надруковані вище. Найважливіше — третій: він дав
**негативний** результат, і це нормальна частина роботи. Модель, яка не бачила
пари «велике кільце», не склала її з двох знайомих слів; вона впевнено каже
«велике» й помиляється у формі.

Це не вада нашої іграшки. Композиційність — відкрита проблема великих
контрастних моделей, і саме тому для неї придумали окремі бенчмарки.

In [ ]:
print("зошит виконано за %.0f секунд" % (time.perf_counter() - notebook_started))

---

## Завдання

### 🟢 Рівень 1 — База

Додай до генератора **сьомий параметр** — наприклад, «з шумом» проти «чіткий»
(достатньо взяти два різні σ) — і додай відповідне слово в описи й у словник
основ. Наскільки точним буде нульовий постріл про цей новий параметр?

**Зроблено, якщо:** надрукована точність нульового пострілу про новий параметр
на трьох зернах, з розкидом, і поруч названий рівень вгадування.

### 🟡 Рівень 2 — Плюс

Побудуй **пошук за текстом**: напиши довільний опис («велике кільце ліворуч») і
дістань пʼять найсхожіших зображень із тестового набору. Намалюй їх.

**Зроблено, якщо:** для трьох різних запитів показано по пʼять картинок і
надруковано, яка частка з них справді відповідає всім трьом параметрам запиту.

### 🔴 Рівень 3 — Виклик

Перевір композиційність на **власному розбитті**. Прибери з навчання іншу
комбінацію — не «велике кільце», а, скажімо, «маленький хрест» або «трикутник
ліворуч» — і зроби висновок **числом**.

**Зроблено, якщо:** надруковано точність на прибраній комбінації для трьох
зерен, поруч — точність на бачених комбінаціях і рівень вгадування, і зроблено
явний висновок одного з двох видів: «композиційність виявлено, точність X проти
рівня вгадування Y» або «композиційності не виявлено: X не відрізняється від
рівня вгадування Y». Обидва висновки однаково правильні як результат.